# AnnData interoperability

**R equivalent:** None — this is a Python-specific feature enabling integration
with the single-cell ecosystem (scanpy, scVI, etc.).

## Layout

| AnnData slot | pyloseq equivalent         | Notes                                |
|--------------|--------------------------|--------------------------------------|
| `X`          | `otu_table` (transposed) | samples as rows (AnnData convention) |
| `obs`        | `sample_data`            | per-sample metadata                  |
| `var`        | `tax_table`              | per-taxon annotation                 |
| `uns["phy_tree"]` | `phy_tree`          | Newick string                        |
| `uns["refseq"]`   | `refseq`            | `{taxon: sequence_str}` dict         |

In [ ]:
import numpy as np
import pyloseq
from pyloseq import Phyloseq, OtuTable, SampleData, TaxTable, PhyTree
from pyloseq.testing.fixtures import load_global_patterns_reference

ref = load_global_patterns_reference()
gp = Phyloseq(
    otu=OtuTable(ref["otu_table"], taxa_are_rows=True),
    sam=SampleData(ref["sample_data"]),
    tax=TaxTable(ref["tax_table"]),
    tree=PhyTree.from_newick(ref["phy_tree_newick"]),
)
print(gp)

## Convert to AnnData

In [ ]:
ad = gp.to_anndata()
print(ad)
print(f"\nX shape (samples × taxa): {ad.X.shape}")
print(f"obs columns: {list(ad.obs.columns)}")
print(f"var columns: {list(ad.var.columns)}")
print(f"uns keys: {list(ad.uns.keys())}")

In [ ]:
# Validate layout
assert ad.X.shape == (gp.nsamples, gp.ntaxa), "X must be samples × taxa"
assert list(ad.obs_names) == list(gp.sample_names)
assert list(ad.var_names) == list(gp.taxa_names)
assert "phy_tree" in ad.uns
print("✓ AnnData layout correct")

## Round-trip back to Phyloseq

In [ ]:
gp2 = Phyloseq.from_anndata(ad)
print(gp2)

assert gp2.ntaxa == gp.ntaxa
assert gp2.nsamples == gp.nsamples
assert gp2.phy_tree is not None
assert gp2.tax_table is not None
assert gp2.sample_data is not None
print("✓ Round-trip preserves all components")

## Use with scanpy (if installed)

In [ ]:
try:
    import scanpy as sc
    # AnnData from pyloseq plugs directly into scanpy workflows
    sc.pp.normalize_total(ad, target_sum=1e4)
    sc.pp.log1p(ad)
    print("✓ scanpy pipeline ran on pyloseq AnnData")
except ImportError:
    print("scanpy not installed — skipping (pip install scanpy)")